In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


Sun Jan  4 11:10:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install --upgrade unsloth transformers datasets accelerate peft trl bitsandbytes sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.2/378.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.8/293.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving JSON_PDF.csv to JSON_PDF.csv


In [ ]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict

SEED = 3407
DATA_PATH = "/content/JSON_PDF.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# MT: src_lang -> tgt_lang (src_lang là tiếng Trung, tgt_lang là tiếng Việt)
need_cols = ["src_lang", "tgt_lang"]
df = df[need_cols].dropna()
df["src_lang"] = df["src_lang"].astype(str).str.strip()
df["tgt_lang"] = df["tgt_lang"].astype(str).str.strip()
df = df[(df["src_lang"] != "") & (df["tgt_lang"] != "")].reset_index(drop=True)

dataset = Dataset.from_pandas(df)

split_80_20 = dataset.train_test_split(test_size=0.2, seed=SEED, shuffle=True)
split_10_10 = split_80_20["test"].train_test_split(test_size=0.5, seed=SEED, shuffle=True)

ds = DatasetDict({
    "train": split_80_20["train"],
    "validation": split_10_10["train"],
    "test": split_10_10["test"],
})

print(ds)
print("Sizes:", {k: len(v) for k, v in ds.items()})


DatasetDict({
    train: Dataset({
        features: ['src_lang', 'tgt_lang'],
        num_rows: 25769
    })
    validation: Dataset({
        features: ['src_lang', 'tgt_lang'],
        num_rows: 3221
    })
    test: Dataset({
        features: ['src_lang', 'tgt_lang'],
        num_rows: 3222
    })
})
Sizes: {'train': 25769, 'validation': 3221, 'test': 3222}


In [ ]:
import os
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True

hf_token = os.environ.get("HF_TOKEN", None)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
    token = hf_token,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

SYSTEM_PROMPT = (
    "Bạn là hệ thống dịch máy."
    "Hãy dịch câu gốc tiếng Trung sang tiếng Việt."
    "Chỉ xuất ra duy nhất bản dịch tiếng Việt, không thêm giải thích."
)

def make_messages(src_text: str, tgt_text: str | None = None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Text:\n{src_text}\n\nDịch câu tiếng Trung trên sang tiếng Việt."},
    ]
    if tgt_text is not None:
        messages.append({"role": "assistant", "content": tgt_text})
    return messages


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.10: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:
def format_for_sft(example):
    msgs = make_messages(example["src_lang"], example["tgt_lang"])
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_sft = ds["train"].map(format_for_sft, remove_columns=ds["train"].column_names)
val_sft   = ds["validation"].map(format_for_sft, remove_columns=ds["validation"].column_names)

print(train_sft[0]["text"][:500])


Map:   0%|          | 0/25769 [00:00<?, ? examples/s]

Map:   0%|          | 0/3221 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Bạn là hệ thống dịch máy.Hãy dịch câu gốc tiếng Trung sang tiếng Việt.Chỉ xuất ra duy nhất bản dịch tiếng Việt, không thêm giải thích.<|eot_id|><|start_header_id|>user<|end_header_id|>

Text:
- 请把经过解冻的鸡肉储藏在冰箱冷冻室中温度最低的地方。这样能使鸡肉储藏更长的时间而不受细菌的侵害。[4] 研究来源

Dịch câu tiếng Trung trên sang tiếng Việt.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

- Bảo quản gà rã đông ở n


In [ ]:
import re
from tqdm.auto import tqdm
from sacrebleu.metrics import BLEU, CHRF, TER

bleu_metric = BLEU(tokenize="13a")
chrf_metric = CHRF()
ter_metric  = TER()

def compute_mt_metrics(preds, refs):
    bleu = bleu_metric.corpus_score(preds, [refs]).score
    chrf = chrf_metric.corpus_score(preds, [refs]).score
    ter  = ter_metric.corpus_score(preds, [refs]).score  # lower is better
    exact = sum(p.strip() == r.strip() for p, r in zip(preds, refs)) / max(1, len(refs))
    return {"BLEU": bleu, "chrF": chrf, "TER": ter, "ExactMatch": exact}

def looks_like_chinese(text: str) -> bool:
    return len(re.findall(r"[\u4e00-\u9fff]", text)) > 0

def make_messages_retry(src_text: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Text:\n{src_text}\n\n"
            "Hãy dịch câu tiếng Trung trên sang tiếng Việt và chỉ trả về kết quả là tiếng Việt."
        )},
    ]

@torch.no_grad()
def generate_predictions(model, tokenizer, dataset, batch_size=4, max_new_tokens=128, limit=None, retry_if_copy=True):
    FastLanguageModel.for_inference(model)

    if limit is not None:
        dataset = dataset.select(range(min(limit, len(dataset))))

    eos_ids = [tokenizer.eos_token_id]
    try:
        eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot_id, int) and eot_id != tokenizer.unk_token_id:
            eos_ids.append(eot_id)
    except Exception:
        pass

    preds, refs, srcs_all = [], [], []

    for i in tqdm(range(0, len(dataset), batch_size)):
        batch = dataset[i:i+batch_size]
        srcs = batch["src_lang"]
        tgts = batch["tgt_lang"]
        srcs_all.extend(srcs)
        refs.extend(tgts)

        prompts = [
            tokenizer.apply_chat_template(
                make_messages(s, None),
                tokenize=False,
                add_generation_prompt=True
            )
            for s in srcs
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            use_cache=True,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.eos_token_id,
        )

        gen_ids = out[:, inputs["input_ids"].shape[1]:]
        texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        texts = [t.strip() for t in texts]

        # Retry nếu model copy tiếng Trung
        if retry_if_copy:
            fixed = []
            for s, t in zip(srcs, texts):
                if looks_like_chinese(t):
                    rp = tokenizer.apply_chat_template(
                        make_messages_retry(s),
                        tokenize=False,
                        add_generation_prompt=True
                    )
                    rin = tokenizer([rp], return_tensors="pt", padding=True,
                                    truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
                    rout = model.generate(
                        **rin,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        temperature=0.0,
                        top_p=1.0,
                        use_cache=True,
                        eos_token_id=eos_ids,
                        pad_token_id=tokenizer.eos_token_id,
                    )
                    rid = rout[:, rin["input_ids"].shape[1]:]
                    t2 = tokenizer.batch_decode(rid, skip_special_tokens=True)[0].strip()
                    fixed.append(t2)
                else:
                    fixed.append(t)
            texts = fixed

        preds.extend(texts)

    return preds, refs, srcs_all


In [ ]:
import pandas as pd

MAX_EVAL_SAMPLES = None

base_preds, test_refs, test_srcs = generate_predictions(
    model, tokenizer, ds["test"],
    batch_size=16,
    max_new_tokens=128,
    limit=MAX_EVAL_SAMPLES,
)

print("Pre-trained model metrics:", compute_mt_metrics(base_preds, test_refs))

for i in range(10):
    print("="*100)
    print("SRC:", test_srcs[i])
    print("REF:", test_refs[i])
    print("PRED:", base_preds[i])

res_base = pd.DataFrame({"src_lang": test_srcs, "ref_vi": test_refs, "pred_vi": base_preds})
res_base.to_csv("/content/test_predictions_base.csv", index=False, encoding="utf-8-sig")
print("Saved:", "/content/test_predictions_base.csv")

  0%|          | 0/202 [00:00<?, ?it/s]

Pre-trained model metrics: {'BLEU': 23.344301829039257, 'chrF': 40.879968394820736, 'TER': 72.1889860408038, 'ExactMatch': 0.04407200496585972}
SRC: 检查所有包含“However”的句子，确保句子是完整的。[8] 研究来源
REF: Hãy kiểm tra tất cả các câu có however để đảm bảo rằng chúng hoàn chỉnh.[8] Nguồn nghiên cứu
PRED: Kiểm tra tất cả các câu có chứa "Tuy nhiên" và đảm bảo các câu là hoàn chỉnh. [8] Nguồn nghiên cứu
SRC: 1. 写下题目。像这样：
REF: 1. Viết phép tính ra như sau:
PRED: 1. Viết câu hỏi. như vậy:
SRC: 2. 在开始菜单中输入iexpress。这会搜索并找到iexpress命令。
REF: 2. Gõ iexpress vào Start để tìm kiếm với lệnh iexpress.
PRED: 2. Trong menu bắt đầu, nhập iexpress. Điều này sẽ tìm kiếm và tìm thấy lệnh iexpress.
SRC: 小提示
REF: Lời khuyên
PRED: Gợi ý
SRC: 有时是随机的。
REF: Đôi khi đó là ngẫu nhiên.
PRED: Có khi là ngẫu nhiên.
SRC: 部分 2
REF: Phần 2
PRED: Phần 2
SRC: 这些神经内分泌肿瘤中的许多会产生对患者有问题的激素。
REF: Rất nhiều trong số các tế bào thần kinh này, một số khối u nội tiết thần kinh, chúng tạo ra hormone có thể gây hại cho bệnh nhân.
PRED: Những khối u

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

cfg = SFTConfig(
    output_dir = "/content/llama3_1_8b_lora_out",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    learning_rate = 2e-4,
    warmup_ratio = 0.03,
    max_steps = 400,
    logging_steps = 10,
    save_steps = 200,
    save_total_limit = 2,

    # Early stopping
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,

    eval_strategy = "steps",
    eval_steps = 200,
    per_device_eval_batch_size = 4,
    fp16 = True,
    fp16_full_eval = True,
    eval_accumulation_steps = 4,

    optim = "adamw_8bit",
    weight_decay = 0.008,
    lr_scheduler_type = "linear",
    seed = SEED,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_sft,
    eval_dataset = val_sft,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    packing = False,
    args = cfg,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3))
trainer.train()

trainer.model.save_pretrained("/content/lora_adapter")
tokenizer.save_pretrained("/content/lora_adapter")
print("Saved adapter:", "/content/lora_adapter")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.12.10 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/25769 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/3221 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=6):   0%|          | 0/25769 [00:00<?, ? examples/s]

...

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,769 | Num Epochs = 1 | Total steps = 400
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
200,0.987200,0.994821
400,0.922200,0.960683


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Saved adapter: /content/lora_adapter


In [ ]:
ft_preds, ft_refs, ft_srcs = generate_predictions(
    trainer.model, tokenizer, ds["test"],
    batch_size=16,
    max_new_tokens=128,
    limit=MAX_EVAL_SAMPLES,
)

print("Fine-tuned model metrics:", compute_mt_metrics(ft_preds, ft_refs))

for i in range(min(5, len(ft_preds))):
    print("="*100)
    print("SRC:", ft_srcs[i])
    print("REF:", ft_refs[i])
    print("PRED:", ft_preds[i])

res_ft = pd.DataFrame({"src_lang": ft_srcs, "ref_vi": ft_refs, "pred_vi": ft_preds})
res_ft.to_csv("/content/test_predictions_ft.csv", index=False, encoding="utf-8-sig")
print("Saved:", "/content/test_predictions_ft.csv")

  0%|          | 0/202 [00:00<?, ?it/s]

Fine-tuned model metrics: {'BLEU': 34.363866664810054, 'chrF': 50.81038141700635, 'TER': 56.728409265224734, 'ExactMatch': 0.16201117318435754}
SRC: 检查所有包含“However”的句子，确保句子是完整的。[8] 研究来源
REF: Hãy kiểm tra tất cả các câu có however để đảm bảo rằng chúng hoàn chỉnh.[8] Nguồn nghiên cứu
PRED: Kiểm tra các câu có chứa từ "However" để đảm bảo rằng chúng là câu hoàn chỉnh.[8] Nguồn nghiên cứu
SRC: 1. 写下题目。像这样：
REF: 1. Viết phép tính ra như sau:
PRED: 1. Viết câu hỏi. Ví dụ:
SRC: 2. 在开始菜单中输入iexpress。这会搜索并找到iexpress命令。
REF: 2. Gõ iexpress vào Start để tìm kiếm với lệnh iexpress.
PRED: 2. Nhập iexpress vào trình đơn Start để tìm lệnh iexpress.
SRC: 小提示
REF: Lời khuyên
PRED: Lời khuyên
SRC: 有时是随机的。
REF: Đôi khi đó là ngẫu nhiên.
PRED: Thỉnh thoảng thì ngẫu nhiên.
Saved: /content/test_predictions_ft.csv


In [ ]:
from google.colab import files
files.download("/content/test_predictions_base.csv")
files.download("/content/test_predictions_ft.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>